# Reverse Engineering an ASIC

Download the required dependencies.
```bash
%pip install -r requirements.txt ipykernel
```
 Graphviz is optional for extraction and simulation, but it is used throughout this notebook to render `.dot` graphs. Install it by running this command in your terminal (if you already use homebrew)
 ```bash
 brew install graphviz
 ```
 Alternatively, you could upload the dot files to any online .dot viewer.

## 1. Getting Started

GDS files are stored in binary, so we can't just scroll through. Lets open the file in a layout viewer such as KLayout, or an online gds viewer such as [tinytapeout](https://gds-viewer.tinytapeout.com/). A screenshot is attached here:

<img src="artifacts/images/puzzlegds_viz.png" alt="Visualized puzzle.gds" width="400">

Certainly looks interesting, but not too interpretable. Zooming in reveals metal wires, vias (joining wires across layers), and individual transtiors. Fortunately, the file stores objects at the granularity of cell types: it places instances of SKY130 cells such as AND gates, multiplexers, and flip-flops at specified coordiantes. The definitions of all the cell types can be found in the [SKY130 PDK](https://skywater-pdk.readthedocs.io/en/main/contents/libraries.html).

The first plan of attack is then to just see what cell types his gds file contains

In [1]:
!python -m tools.inspect_gds puzzle.gds

library cells: 81
top cell: puzzle
direct instances: 9875
top-level labels: ['I', 'O[0]', 'O[1]', 'O[2]', 'O[3]', 'O[4]', 'O[5]', 'O[6]', 'O[7]', 'clk', 'enable', 'rst_n', 'success', 'VGND', 'VPWR', 'VGND', 'VPWR']

by kind: {'filler': 890, 'via': 8221, 'logic': 728, 'other': 36}

instances by cell type:
   3154  VIA_M1M2_PR  [via]
   2696  VIA_L1M1_PR_MR  [via]
    676  sky130_fd_sc_hd__tapvpwrvgnd_1  [filler]
    618  VIA_via2_3_2000_480_1_6_320_320  [via]
    618  VIA_via4_5_2000_480_1_5_400_400  [via]
    618  VIA_via3_4_2000_480_1_5_400_400  [via]
    333  VIA_M2M3_PR  [via]
    204  sky130_fd_sc_hd__decap_3  [filler]
    108  VIA_via5_6_2000_2000_1_1_1600_1600  [via]
     84  sky130_fd_sc_hd__dfrtp_2  [logic]
     69  VIA_M3M4_PR  [via]
     49  sky130_fd_sc_hd__nor2_2  [logic]
     39  sky130_fd_sc_hd__nand2_2  [logic]
     31  sky130_fd_sc_hd__o21a_2  [logic]
     30  sky130_fd_sc_hd__and2b_2  [logic]
     29  sky130_fd_sc_hd__xnor2_2  [logic]
     26  sky130_fd_sc_hd__a31o_2  

We now know the identity of all the gates (for example, nor2, and2, etc.), the wires, vias, and the inputs and outputs to the circuit as a whole: `clk`, `rst_n`, `enable`, `I`, `O[7:0]`, and `success`. We next need to find out which gates are connected to which.

## 2. Connectivity

The GDS file gives us the location of the pins of each gate and all metal conductors. On each layer, merge touching wires into connected regions. Then merge regions on adjacent layers wherever a via connects them, into one giant conducting net. Next go through every pin on every gate and find out which net it touches, and assemble into a big list of nets and pins.

In [2]:
# Extract nets from the gds. This can take a little while.
!python -m tools.extract_netlist puzzle.gds --json generated/puzzle.json

top cell: puzzle
merging conductor layers...
  li1   raw=13931  merged regions=6680
  met1  raw=14366  merged regions=3001
  met2  raw= 8517  merged regions=2060
  met3  raw= 2547  merged regions=811
  met4  raw=  855  merged regions=45
  met5  raw=  144  merged regions=18
stitching layers through vias...
  li1->met1: 19764 cuts, 19764 stitched
  met1->met2: 6869 cuts, 6869 stitched
  met2->met3: 3423 cuts, 3423 stitched
  met3->met4: 3159 cuts, 3159 stitched
  met4->met5: 108 cuts, 108 stitched
assigning pins to nets...
naming nets from top-level ports...
  unresolved pins: 0
  unresolved ports: 0

=== 741 nets extracted ===

net I  (45 terminals)
    U101_nand2_2.A
    U102_a31o_2.A1
    U103_a31o_2.A1
    U112_nand2_2.A
    U120_a31o_2.A1
    U121_a31o_2.A1
    U123_nand2_2.A
    U125_nand4_2.A
    U126_nand4_2.A
    U132_nand4_2.A
    U148_a31o_2.A1
    U149_nand4_2.A
    U154_nand4_2.A
    U179_nand4_2.A
    U183_a31o_2.A1
    U187_nand2_2.A
    U195_a31o_2.A1
    U199_nand4_2.A
 

In [2]:
# Look at a few extracted nets, skip over the massive VGND and VPWR nets, they just provide power.
import json
from pathlib import Path

ListOfNets = Path('generated/puzzle.json')

raw = json.loads(ListOfNets.read_text())
for name, record in list(raw['nets'].items())[:8]:
    if name in {'VGND', 'VPWR'}: 
        continue
    print(name, json.dumps(record, indent=2))

I {
  "aliases": [
    "I"
  ],
  "terminals": [
    "U101_nand2_2.A",
    "U102_a31o_2.A1",
    "U103_a31o_2.A1",
    "U112_nand2_2.A",
    "U120_a31o_2.A1",
    "U121_a31o_2.A1",
    "U123_nand2_2.A",
    "U125_nand4_2.A",
    "U126_nand4_2.A",
    "U132_nand4_2.A",
    "U148_a31o_2.A1",
    "U149_nand4_2.A",
    "U154_nand4_2.A",
    "U179_nand4_2.A",
    "U183_a31o_2.A1",
    "U187_nand2_2.A",
    "U195_a31o_2.A1",
    "U199_nand4_2.A",
    "U200_nand2_2.A",
    "U209_a31o_2.A1",
    "U212_a31o_2.A1",
    "U219_nand4_2.A",
    "U227_nand4_2.A",
    "U229_nand4_2.A",
    "U254_a21boi_2.A1",
    "U275_o21ai_2.A1",
    "U376_a31o_2.A1",
    "U383_mux2_1.A1",
    "U419_or2_2.B",
    "U421_nand2_2.B",
    "U439_a32o_2.A1",
    "U446_a21oi_2.A1",
    "U447_and3_2.B",
    "U4_a31o_2.A1",
    "U595_a31o_2.A1",
    "U601_nand2_2.A",
    "U606_a31o_2.A1",
    "U609_nand4_2.A",
    "U611_nand4_2.A",
    "U617_a31o_2.A1",
    "U623_nand4_2.A",
    "U631_nand2_2.A",
    "U633_nand2_2.A",
    "U

A record such as `U119_clkbuf_4.A` means pin `A` of a particular clock-buffer instance touches that net. Cell model definitions from the official SKY130 Liberty data tell us pin directions and the Boolean functions they implement. If all is well, each net should have a single pin driving it, and all other pins receiving that as input. We can form a directed bipartite graph: gate outputs drive net nodes, and net nodes fan out to gate inputs - while sanity checking that each net have exactly one driver.

In [4]:
!.venv/bin/python -m tools.analyze_netlist {ListOfNets} summary

Source:          generated/puzzle.json
Instances:       728
Nets:            741 (739 non-power)
Terminals:       4223
State elements:  92
Shift registers: 1 strictly verified

Ports (direction inferred from cell pin directions):
  I                input             drivers=0  loads=45
  O[0]             output            drivers=1  loads=0
  O[1]             output            drivers=1  loads=0
  O[2]             output            drivers=1  loads=0
  O[3]             output            drivers=1  loads=0
  O[4]             output            drivers=1  loads=0
  O[5]             output            drivers=1  loads=0
  O[6]             output            drivers=1  loads=0
  O[7]             output            drivers=1  loads=0
  VGND             power             drivers=0  loads=0
  VPWR             power             drivers=0  loads=0
  clk              input             drivers=0  loads=1
  enable           input             drivers=0  loads=1
  rst_n            input             driv

So we have 728 logical instances, 741 nets, 92 memory elements (flip-flops), and one suspicious floating internal net (no driver), `n0550`...to be investigated later.

### Lets just draw everything

We have the whole connectivity graph...lets render it out. One subtlety: to keep things streamlined, we recursively work backwards from our ultimate goal (making success = 1), and render only those gates that could possibly influence the value of success - the "backwards cone" influencing the success net. The code to do this is in `tools/analyze_netlist.py`.

<img src="graphviz.svg" alt="Backwards cone of success" width="950">

If you want to render this yourself, the code is provided in the next cell. There's no way we're making sense of this with the eyeball test. A tiny "success" node is visible at the top right. We must smartly query this graph in small sections to try and infer its function.

In [13]:
# Reproduce the  graph shown above.
!python -m tools.analyze_netlist {ListOfNets} cone success --through-flops --dot generated/puzzle-success-backwards-cone.dot
!dot -Tsvg generated/puzzle-success-backwards-cone.dot -o generated/puzzle-success-backwards-cone.svg

Target net:         success
Cells in cone:      468
Combinational:      389
State boundaries:   0
Nets visited:       470


Combinational cells:
     34  nor2
     30  nand2
     28  o21a
     23  a31o
     20  and2b
     20  mux2
     17  inv
     17  nand2b
     14  a21o
     14  and3
     14  nand4
     13  and2
     12  a21oi
     12  and4bb
     12  xnor2
     12  xor2
     10  or4
      9  o211a
      9  or2
      8  and4
      8  or3
      8  or4b
      4  a32o
      3  a211o
      3  a22o
      3  and4b
      3  o21ai
      2  a21bo
      2  a221o
      2  nor4
      2  o22ai
      2  o311a
      2  o31ai
      2  o32a
      1  a211oi
      1  a221oi
      1  a311o
      1  a41oi
      1  buf
      1  conb
      1  nand3
      1  nand3b
      1  nor3
      1  o211ai
      1  o21ba
      1  o221a
      1  o22a
      1  o2bb2a
      1  or4bb
Wrote generated/puzzle-success-backwards-cone.dot
Render with: dot -Tsvg generated/puzzle-success-backwards-cone.dot -o generated/puzzle-suc

## 3. Rehearse the method on the warm-up

Before interpreting 92 state bits, test the entire pipeline on something small. The warm-up contains two eight-bit enabled shift registers. Once those are recognized, `success` is a combinational function of only sixteen current-state bits. Enumerating all `2^16 = 65,536` assignments is entirely reasonable.

In [ ]:
!.venv/bin/python -m tools.analyze_netlist artifacts/netlists/warmup.json shift-registers
!.venv/bin/python -m examples.warmup_exhaustive

The successful warm-up states all satisfy `A + B = 496`. This is a good first victory because it validates extraction, pin direction, Boolean evaluation, bit ordering, and the shift-register abstraction independently.

Trying the same trick on the puzzle would mean `2^92` current states—way too many. Exhaustive enumeration of all the possible states the memory elements could take simply won't work. The warm-up exercise does, however, give us the idea to look for and abstract away shift registers in the main puzzle into their own block - gathering up clutter and making it interpretable.

## 4. Take some hints - the sample inputs file

It is time to analyze the provided `example_inputs.vcd` file. It instructs us to look at it through a waveform viewer, we can use an [online vcd viewer](https://app.surfer-project.org/). A biref section of it looks like this:

<img src="artifacts/images/vcd_view.png" alt="VCD view" width="800">

Success is always 0, but on application of this particular input, and setting the enable to 0, results in some output. With some educated guesswork, we can try seeing what ASCII characters this output corresponds to, and we find that it says "TRY AGAIN". Which (and how many) input bits did we feed in the first place?

In [3]:
def inputs_at_rising_edge(path):
    names = {'!': 'clk', '"': 'rst_n', '#': 'enable', '$': 'I'}
    values = {symbol: 0 for symbol in names}
    samples = []
    time = 0
    for raw_line in Path(path).read_text().splitlines():
        line = raw_line.strip()
        if line.startswith('#') and line[1:].isdigit():
            time = int(line[1:])
        elif len(line) == 2 and line[0] in '01' and line[1] in values:
            symbol = line[1]
            old = values[symbol]
            values[symbol] = int(line[0])
            if symbol == '!' and old == 0 and values[symbol] == 1:
                samples.append({'time': time, **{names[s]: values[s] for s in names}})
    return samples # all values of inputs at rising clock edge.

samples = inputs_at_rising_edge('example_inputs.vcd')
attempts, current = [], []
for sample in samples:
    if sample['enable']:
        current.append(bool(sample['I']))
    elif current:
        attempts.append(current)
        current = []
if current:
    attempts.append(current)

print('Input string length for each "attempt":', [len(bits) for bits in attempts])

for index, bits in enumerate(attempts):
    print("Input ", index, ":")
    print(''.join('1' if bit else '0' for bit in bits))

Input string length for each "attempt": [121, 121]
Input  0 :
0010101000000010110000101001100000000010000001110110000100101100001110011000000010110000001011100000000010000011001110000
Input  1 :
1101011000010011110000000001000001000011000011101110000100001100001001011000000101110000110011100000000010000000000100000


Okay! Each input attempt appears to be 121 bits long. The actual inputs bits are also dumped into the output, can we possibly make sense of them? Lets try 11 groups of 11, and convert the 11-bit long strings into a decimal number. With a little trial and error, turns out they are ASCII characters encoded little-endian style (the first bit into the circuit is the most significant bit). Lets decode it:

In [4]:
def decode_eleven_bit_words(bits):
    values = [
        sum(bit << offset for offset, bit in enumerate(bits[start:start + 11]))
        for start in range(0, len(bits), 11)
    ]
    return values, ''.join(chr(value) for value in values)

for index, bits in enumerate(attempts, 1):
    values, text = decode_eleven_bit_words(bits)
    print(index, values, repr(text))

1 [84, 104, 101, 32, 110, 105, 103, 104, 116, 32, 115] 'The night s'
2 [107, 121, 32, 97, 119, 97, 105, 116, 115, 32, 32] 'ky awaits  '


**“The night sky awaits”**! 
Great. Now, a natural thing to do is to treat this like a black box and write code that would allow us to send in a 121 bit word, simulate the whole circuit, and see what the value of success and what the outputs are.

## 5. Simulate

We want a function that takes a 121-bit word, clocks it into the chip, and tells us what `success` and `O[7:0]` do.

The circuit is synchronous: on a rising clock edge every flip-flop samples its D pin, and all Qs update together. Gates in between are just the boolean functions from the SKY130 cell library. So a "simulation" is: remember the current Qs, evaluate every D, write the new Qs.

Most of the 92 flops are `dfrtp` (on reset, Q = 0). Four are `dfstp`: reset *sets* them to 1. Four more are `dfxtp` and have no reset pin at all.

There's also that floating net `n0550`. For now force it to 0 - since it doesn't influence success anyway, we'll investigate it at the end.


In [5]:
from pathlib import Path

from tools.netlist_ir import Design
from tools.circuit_eval import CircuitEvaluator
from tools.play import Play, bits_at, show_grid

ListOfNets = Path('generated/puzzle.json')
if not ListOfNets.exists():
    ListOfNets = Path('artifacts/netlists/puzzle.json')

GENERATED = Path('generated')
GENERATED.mkdir(exist_ok=True)

design = Design.load(ListOfNets)
sim = Play(design)  # reset / tick / scan / replay, with n0550 forced to 0
print('loaded', ListOfNets)
print('flip-flops:', len(sim.state_net))
print('success is driven by:', [t.name for t in design.resolve_net('success').drivers])


loaded generated/puzzle.json
flip-flops: 92
success is driven by: ['U28_dfrtp_2.Q']


In [6]:
q = sim.q
reset_state = sim.reset_state
tick = sim.tick
scan = sim.scan
replay_attempt = sim.replay_attempt
state_net = sim.state_net

for index, bits in enumerate(attempts, 1):
    byte_values, success_values = replay_attempt(bits)
    text = bytes(byte_values).split(b'\0', 1)[0].decode('ascii')
    print(f'attempt {index}: {text!r}  success={any(success_values)}')


attempt 1: 'TRY AGAIN'  success=False
attempt 2: 'TRY AGAIN'  success=False


Our sanity check worked, replaying the inputs that were played in `example_inputs.vcd` produces the same output. Now we can start opening the box. We'll attempt to slowly abstract groups of flip-flops into functions we understand; hopefully after enough passes things will start to make sense.


## 6. Shift Registers

Taking inspiration from the warm-up; if `I` is a 121-bit serial stream, it probably lands in a shift register: a mux in front of a D-flop, the mux selecting between "hold Q" and "take the previous bit," repeated down a chain. We hunt for that wiring and find a 12-bit shift register.

In [7]:
!.venv/bin/python -m tools.analyze_netlist {ListOfNets} shift-registers --dot generated/puzzle-shift-register-block.dot
!dot -Tsvg generated/puzzle-shift-register-block.dot -o generated/puzzle-shift-register-block.svg


Number of shift registers: 1

shift_I: ShiftRegister[12]
  serial input: I
  enable:       n0005
  clock:        clk
  members:      12 mux, 12 flip-flops

Wrote generated/puzzle-shift-register-block.dot
Render with: dot -Tsvg generated/puzzle-shift-register-block.dot -o generated/puzzle-shift-register-block.svg


<img src="generated/puzzle-shift-register-block.svg" alt="shift_I and the four Q bits that leave it" width="560">


In [1]:
sr = design.strict_shift_registers().shift_registers[0]
print('bits that leave the block:')
for index, stage in enumerate(sr.stages):
    loads = sr.external_q_loads[stage.q_net]
    if loads:
        print(f'  Q[{index:2}]  {stage.q_net}  ->  {", ".join(loads)}')


bits that leave the block:
  Q[ 0]  n0342  ->  U360_a22o_2.A2
  Q[ 9]  n0364  ->  U374_a221o_2.A2
  Q[10]  n0366  ->  U374_a221o_2.B2
  Q[11]  n0344  ->  U360_a22o_2.B2


A 12-bit window on `I`. Eight of those bits only feed the next stage; four leave the box — stages 0, 9, 10, and 11 — and land on two combinational gates. Those four taps are delays of 1, 10, 11, and 12 input bits. If you already believe in an 11-wide grid you can start muttering "neighbors," but we do **not** have a coordinate system from the hardware yet. So we will not name them neighbors.

The twelve muxes and twelve flip-flops are now one box:

```text
I, n0005, clk  -->  [ shift_I : ShiftRegister[12] ]  -->  Q[0], Q[9], Q[10], Q[11]
```

`n0005` is not the top-level `enable` pin. Something computes "should the window actually slide?" We'll find that something when we meet the machine that knows how far through the 121 bits we are.

Eighty flip-flops to go.


## 7. Don't stare at 80 flops. Cut them into machines.

For every remaining flop, walk backwards from its D pin until you hit another flop's Q. If `A.Q` shows up in the next-state logic of `B`, draw an edge `A → B`. Then collapse every strongly connected component to a single blob.

A counter or a little state machine stays strongly connected even after synthesis has replaced a mux with a pile of AOI gates. A shift register does *not* become one giant SCC — each stage only depends on the previous one, so you get a chain of 1s (which we already boxed). That's the distinction we want.


In [ ]:
from collections import Counter, defaultdict

seq = {
    inst.name: inst
    for inst in design.instances.values()
    if inst.sequential
}
shift_ffs = {stage.flip_flop for stage in sr.stages}

deps = {
    name: set(design.backward_cone(inst.pins['D'].net).state_boundaries)
    for name, inst in seq.items()
}

def strongly_connected_components(graph):
    index = 0
    stack, on_stack = [], set()
    indices, lowlinks, result = {}, {}, []

    def visit(node):
        nonlocal index
        indices[node] = lowlinks[node] = index
        index += 1
        stack.append(node)
        on_stack.add(node)
        for neighbor in graph[node]:
            if neighbor not in indices:
                visit(neighbor)
                lowlinks[node] = min(lowlinks[node], lowlinks[neighbor])
            elif neighbor in on_stack:
                lowlinks[node] = min(lowlinks[node], indices[neighbor])
        if lowlinks[node] == indices[node]:
            component = []
            while True:
                member = stack.pop()
                on_stack.remove(member)
                component.append(member)
                if member == node:
                    break
            result.append(component)

    for node in graph:
        if node not in indices:
            visit(node)
    return result

sccs = strongly_connected_components(deps)
print('SCC size histogram:', dict(sorted(Counter(map(len, sccs)).items())))
print()
print('multi-bit SCCs (shift_I hidden):')
for component in sorted(sccs, key=lambda c: (-len(c), sorted(c)[0])):
    if len(component) == 1:
        continue
    if set(component) <= shift_ffs:
        continue
    types = Counter(seq[name].cell_type for name in component)
    print(f'  size {len(component):2}  {dict(types)}  e.g. {sorted(component)[0]}')


In [ ]:
# Split the 2-bit machines by how much combinational logic sits in front of them.
two_bit = [set(c) for c in sccs if len(c) == 2]
thin, fat, reused = [], [], []
for pair in two_bit:
    comb_cells = set()
    extra = set()
    has_mux = False
    for name in pair:
        cone = design.backward_cone(seq[name].pins['D'].net)
        extra |= cone.state_boundaries - pair
        for cell in cone.cells:
            inst = design.instances[cell]
            if inst.sequential:
                continue
            comb_cells.add(cell)
            if inst.cell_type == 'mux2':
                has_mux = True
    info = (pair, len(comb_cells), frozenset(extra), has_mux)
    if has_mux and len(comb_cells) < 30:
        reused.append(info)
    elif len(comb_cells) < 30:
        thin.append(info)
    else:
        fat.append(info)

print(f'tiny 2-bit SCCs (a handful of gates):     {len(thin)}')
print(f'tiny 2-bit SCC that also has a mux:       {len(reused)}')
print(f'fat  2-bit SCCs (~150 gates each):        {len(fat)}')
if fat:
    fat_comb = []
    for pair, *_ in fat:
        cells = set()
        for name in pair:
            cells |= {
                c for c in design.backward_cone(seq[name].pins['D'].net).cells
                if not design.instances[c].sequential
            }
        fat_comb.append(cells)
    shared_comb = set.intersection(*fat_comb)
    union_comb = set.union(*fat_comb)
    print(f'  shared combinational cells among fat ones: {len(shared_comb)} / {len(union_comb)}')

print()
print('the mux-y pair:', sorted(reused[0][0]) if reused else None)

loners = [
    c[0] for c in sccs
    if len(c) == 1 and c[0] not in shift_ffs
]
print(f'singletons outside shift_I: {len(loners)}')
print('  ', ', '.join(sorted(loners)))
print()
print('and success is literally this flop\'s Q:', [t.name for t in design.resolve_net('success').drivers])


In [ ]:
overview = r'''
digraph blocks {
  rankdir=LR;
  graph [fontname=Helvetica, labelloc=t, label="92 flops after collapsing shift_I and SCCs"];
  node [fontname=Helvetica, shape=box];
  I [shape=diamond];
  en [shape=diamond, label="enable"];
  suc [shape=diamond, label="success"];
  O [shape=diamond, label="O[7:0]"];

  shift [shape=box3d, label="shift_I\nShiftRegister[12]"];
  hub [label="9-bit SCC\nnever depends on I"];
  pairs [label="23 × 2-bit SCCs\n11 tiny + 1 mux-y + 11 fat"];
  chain [label="8 singleton flops\nin a chain, not an SCC"];
  taps [label="singleton\nD-cone includes the 4 taps"];
  other [label="a few other\nsticky singletons"];
  pos [label="4-bit SCC\nno reset pin"];
  msg [label="8-bit SCC\n4× async-set + 4× dfrtp"];
  U28 [label="U28\n(drives success)"];

  I -> shift;
  en -> hub;
  hub -> shift [label="n0005?"];
  shift -> taps [label="Q[0,9,10,11]"];
  hub -> pairs;
  hub -> chain;
  hub -> taps;
  I -> pairs;
  I -> chain;
  pairs -> U28;
  chain -> U28;
  taps -> U28;
  other -> U28;
  U28 -> suc;
  hub -> pos;
  pos -> msg;
  msg -> O;
}
'''
(GENERATED / 'blocks-structural.dot').write_text(overview)
!dot -Tsvg generated/blocks-structural.dot -o generated/blocks-structural.svg


<img src="generated/blocks-structural.svg" alt="structural block overview" width="950">

That's the whole trick. Ninety-two bits became a dozen boxes. We still don't know what most of the boxes *do* — and that's fine. Naming comes from asking each box a question small enough that it has nowhere to hide.

The 9-bit SCC is the right first victim: it is the only chunky machine whose next-state logic never mentions `I`. Stars on a grid shouldn't move the cursor. If this chip is scanning 121 cells, the thing that knows *which* cell we're on has to ignore the data pin.


## 8. The one machine that ignores I

Hold `I` at 0, clock `enable` for a while, and print those nine bits. No hypothesis required — just read.


In [ ]:
HUB = [
    'U342_dfrtp_2', 'U344_dfrtp_2', 'U346_dfrtp_2', 'U347_dfrtp_2', 'U354_dfrtp_2',
    'U458_dfrtp_2', 'U459_dfrtp_2', 'U462_dfrtp_2', 'U466_dfrtp_2',
]

def pack(state, names):
    return ''.join(str(q(state, name)) for name in names)

state = reset_state()
print('t   ' + ' '.join(name[1:4] for name in HUB) + '   bits')
for t in range(125):
    interesting = t <= 23 or t in {33, 44, 55, 88, 110, 119, 120, 121, 122} or t % 11 == 0
    if interesting:
        print(f'{t:3}  {pack(state, HUB)}')
    enable = t < 121
    state = tick(state, enable, False).next_state


Reading down the columns:

- `U347` flips every clock. `U354` every two, `U344` every four, `U346` every eight — until they all wrap at 11. That's a 0…10 counter. Call it **col**.
- Every eleven clocks, `U462` / `U466` / `U459` / `U458` do the same trick. That's a 0…10 counter that ticks once per wrap. Call it **row**.
- At clock 121, `U342` goes high and everything else freezes. Call it **done**.

The chip is walking an 11×11 board. The night-sky hint was cute; this is the hardware agreeing.


In [ ]:
COL_BITS = ['U347_dfrtp_2', 'U354_dfrtp_2', 'U344_dfrtp_2', 'U346_dfrtp_2']  # LSB first
ROW_BITS = ['U462_dfrtp_2', 'U466_dfrtp_2', 'U459_dfrtp_2', 'U458_dfrtp_2']
DONE = 'U342_dfrtp_2'

def decode(state, names):
    return sum(q(state, name) << i for i, name in enumerate(names))

state = reset_state()
print('t   col row done  n0005  U26')
for t in range(124):
    ev = CircuitEvaluator(
        design,
        {**state, 'rst_n': True, 'enable': t < 121, 'I': False, 'n0550': False},
    )
    if t <= 12 or t >= 119 or t % 11 == 0:
        print(
            f'{t:3}   {decode(state, COL_BITS):2}  {decode(state, ROW_BITS):2}   '
            f'{q(state, DONE)}     {int(ev.value("n0005"))}     {q(state, "U26_dfrtp_2")}'
        )
    state = tick(state, t < 121, False).next_state

print()
print('n0005 driver:', design.source_description('n0005'))
print('U351 function:', design.instances['U351_and2b_2'].model.function)
print('U351.A_N <-', design.source_description(design.instances['U351_and2b_2'].pins['A_N'].net))
print('U351.B   <-', design.source_description(design.instances['U351_and2b_2'].pins['B'].net))


And the shift-register enable we left hanging is not mysterious at all:

```text
n0005 = enable AND NOT done
```

The window only slides while the scan is live. After bit 121, `done` sticks, `shift_I` holds, and `U26` goes high on the next edge — "stop listening to `I`, start talking on `O`". That's the output-message path (`TRY AGAIN` and friends). Side quest; we can ignore it until `success` makes sense.

We now have permission to talk about rows and columns as *circuit facts*, not as a guess from ASCII art.


## 9. So what are those four taps?

Delays of 1, 10, 11, and 12 on an 11-wide row-major scan are exactly the four neighbors of the current cell that have *already been seen*: left, upper-right, above, upper-left. The current bit is `I` itself. The other neighbors haven't arrived yet.

Which singleton actually listens to those taps? Don't guess names — ask the D-cones. From here on a 121-bit word *is* the board: `scan` clocks it in, `show_grid` draws it.


In [ ]:
tap_ffs = {sr.stages[i].flip_flop for i in (0, 9, 10, 11)}
col_and_done = set(COL_BITS) | {DONE}

print('tap flops:', sorted(tap_ffs, key=lambda n: next(i for i, st in enumerate(sr.stages) if st.flip_flop == n)))
print()
print('singletons whose extra state is only (taps + col + done):')
adj_candidates = []
for name in loners:
    extra = deps[name] - {name}
    if tap_ffs <= extra <= (tap_ffs | col_and_done):
        adj_candidates.append(name)
        print(f'  {name}')
        print(f'    extra = {sorted(extra)}')

ADJ = adj_candidates[0]
print()
print('using', ADJ)


One candidate, `U381`. Its entire world is the four taps, the column counter (so it can tell a wrap-around "distance 10" from a real diagonal), and itself — i.e. a sticky bit. Lets poke it.


In [ ]:
adjacency_tests = {
    'one isolated bit': bits_at(0),
    'horizontal  (distance 1)': bits_at(0, 1),
    'diagonal / wrap-ish (distance 10)': bits_at(1, 11),
    'vertical    (distance 11)': bits_at(0, 11),
    'other diagonal (distance 12)': bits_at(0, 12),
    'distance 10 in the SAME row (should NOT count)': bits_at(0, 10),
}
for label, bits in adjacency_tests.items():
    print(label)
    show_grid(bits)
    state = scan(bits, clocks=13)
    print(f'  {ADJ.split("_")[0]}={q(state, ADJ)}')


Isolated bit: stay 0. Real neighbors, including both diagonals: stick at 1. Positions 0 and 10 are distance 10 *in the same row*, which is not a neighbor — and the column counter lets the gate ignore them. Positions 1 and 11 *are* a diagonal across the row boundary.

So `U381` is an adjacency latch. Stars are not allowed to touch, not even on a corner. Box it: `Adj`.


## 10. Twenty-three little two-bit machines

They all listen to `I` and to the scan controller. Eleven are tiny, eleven are fat and share a giant decoder, and one tiny one has a mux like it wants to be reset and reused. The reused one is the obvious place to start: if col wraps every 11, a *row* counter would too.


In [ ]:
ROW_PAIR = set(reused[0][0])
print('reused pair:', sorted(ROW_PAIR))
print('extra state (should be col+done, not row):', sorted(reused[0][2]))

# A sticky loner whose world is only this pair + the column counter.
row_stickies = []
for name in loners:
    extra = deps[name] - {name}
    if ROW_PAIR <= extra <= (ROW_PAIR | col_and_done):
        row_stickies.append(name)
        print('sticky that sees the pair:', name, 'extra', sorted(extra))

BAD_ROW = row_stickies[0]
ROW_STATE = sorted(ROW_PAIR)
print('row-state bits:', ROW_STATE)
print('bad-row sticky:', BAD_ROW)


In [ ]:
print('stars in first row   state@10   bad-row@11')
for count in range(5):
    # Space them out so Adj does not confuse the experiment.
    bits = bits_at(*range(0, 2 * count, 2))
    mid = scan(bits, clocks=10)
    end = scan(bits, clocks=11)
    state_bits = tuple(q(mid, name) for name in ROW_STATE)
    print(f'  {count}                    {state_bits}         {q(end, BAD_ROW)}')


During the row the two-bit state counts; at the eleventh clock it snaps back for reuse. The sticky error bit stays 0 only when the row contained **exactly two** 1s. We just got a rule without reading a single AOI equation.

The other tiny 2-bit machines should be columns (each one fires at a single `col`). The fat ones, sitting behind that shared decoder of `(row, col)`, should be the irregular regions. Check by flipping `I` at each of the 121 positions and seeing who twitches — 242 transitions, not 121 full simulations.


In [ ]:
thin_pairs = [info[0] for info in thin]
fat_pairs = [info[0] for info in fat]
all_pairs = thin_pairs + [ROW_PAIR] + fat_pairs

hits_by_position = []
positions_by_pair = [set() for _ in all_pairs]
state = reset_state()
common = {'rst_n': True, 'enable': True, 'n0550': False}

for position in range(121):
    low = tick(state, True, False).next_state
    high = tick(state, True, True).next_state
    hits = []
    for index, pair in enumerate(all_pairs):
        if any(low[state_net[name]] != high[state_net[name]] for name in pair):
            hits.append(index)
            positions_by_pair[index].add(position)
    hits_by_position.append(hits)
    state = low

n_thin = len(thin_pairs)
row_index = n_thin  # we appended ROW_PAIR there
col_like = []
for i, positions in enumerate(positions_by_pair[:n_thin]):
    residues = {p % 11 for p in positions}
    print(f'tiny[{i}] hits={len(positions):3}  residues={sorted(residues)}  e.g. {sorted(thin_pairs[i])[0]}')
    if len(positions) == 11 and len(residues) == 1:
        col_like.append(i)

print()
print('row pair hits:', len(positions_by_pair[row_index]))
print('tiny machines that look like columns (11 hits, one residue):', len(col_like))
print('fat machine hit-counts:', [len(positions_by_pair[n_thin + 1 + i]) for i in range(len(fat_pairs))])


In [ ]:
# Label regions 0..10 by the first cell that belongs to them, just so the picture is stable.
fat_order = sorted(
    range(len(fat_pairs)),
    key=lambda i: min(positions_by_pair[n_thin + 1 + i]),
)
fat_label = {index: label for label, index in enumerate(fat_order)}

region_map = []
for row in range(11):
    mapped = []
    for column in range(11):
        position = row * 11 + column
        fat_hits = [h - (n_thin + 1) for h in hits_by_position[position] if h >= n_thin + 1]
        mapped.append(fat_label[fat_hits[0]])
    region_map.append(mapped)

for row in region_map:
    print(' '.join(f'{region:2}' for region in row))


In [ ]:
def write_grid_dot(path, regions, stars=frozenset()):
    colors = [
        '#f4cccc', '#fce5cd', '#fff2cc', '#d9ead3', '#d0e0e3', '#cfe2f3',
        '#d9d2e9', '#ead1dc', '#c9daf8', '#b6d7a8', '#ffe599',
    ]
    lines = [
        'graph grid {', 'layout=neato;', 'overlap=false;', 'splines=false;',
        'node [shape=square, fixedsize=true, width=0.46, height=0.46, fontname=Helvetica, fontsize=16];',
        'edge [color="#999999", penwidth=0.6];',
    ]
    for row in range(11):
        for column in range(11):
            position = row * 11 + column
            label = '*' if position in stars else ''
            color = colors[regions[row][column] % len(colors)]
            lines.append(
                f'n{position} [pos="{column},{10-row}!", label="{label}", '
                f'style=filled, fillcolor="{color}"];'
            )
            if column:
                lines.append(f'n{position-1} -- n{position};')
            if row:
                lines.append(f'n{position-11} -- n{position};')
    lines.append('}')
    Path(path).write_text('\n'.join(lines))

write_grid_dot(GENERATED / 'region-map.dot', region_map)
!neato -n2 -Tsvg generated/region-map.dot -o generated/region-map.svg


<img src="generated/region-map.svg" alt="11 irregular regions on the 11x11 board" width="420">

Eleven columns, eleven rows, eleven irregular contiguous regions. Same 0/1/2/3 experiment on a column machine or a region machine: they also accept exactly two. (Try it if you like; it is the same loop as the row, just wait until that column/region has seen its cells.)

This is now recognizably a Star Battle board — derived from wiring and a handful of traces, not from the slogan on the sample input.


## 11. One more counter, then the AND that is `success`

Eight singleton flops are still sitting there in a chain rather than an SCC. That's what a binary counter looks like: bit *k* depends on bits `0..k`, with no loop back. Find the chain by starting from the loner whose only extra state is `done`, and walking up.


In [ ]:
pop_bits = []
covered = {DONE}
remaining = set(loners) - {ADJ, BAD_ROW, 'U26_dfrtp_2', 'U27_dfrtp_2', 'U28_dfrtp_2'}
while True:
    nxt = [
        name for name in remaining
        if (deps[name] - {name}) <= covered and name not in covered
    ]
    if not nxt:
        break
    nxt.sort(key=lambda name: (len(deps[name]), name))
    pop_bits.append(nxt[0])
    covered.add(nxt[0])
    remaining.remove(nxt[0])

print('population bits LSB -> MSB:')
for i, name in enumerate(pop_bits):
    print(f'  [{i}] {name}  cone {sorted(deps[name])}')

def population(state):
    return sum(q(state, name) << i for i, name in enumerate(pop_bits))

print()
for count in [0, 1, 2, 4, 8, 22, 38]:
    bits = '1' * count + '0' * (121 - count)
    state = scan(bits)
    print(f'injected ones={count:2}  counter={population(state):2}')


It's a population counter. Stream *k* ones, read *k*. The final combinational pile sitting in front of `U28` (whose Q *is* the `success` pin) is almost entirely AND gates over the flags we just named. English, without expanding 47 cells:

```text
done after 121 input clocks
AND  total population = 22
AND  every row count = 2
AND  every column count = 2
AND  every region count = 2
AND  Adj never fired
AND  no row reported a bad count
```

`U28` registers that on the first clock after `enable` falls. Twenty-two is redundant with eleven rows of two, but redundancy is comforting when the alternative is trusting a single cone.

`U27` is the sibling combiner used by the fail / `TRY AGAIN` message path. Same ingredients, different customer.


In [ ]:
named = r'''
digraph named {
  rankdir=LR;
  graph [fontname=Helvetica, labelloc=t, label="named blocks"];
  node [fontname=Helvetica, shape=box];
  I [shape=diamond];
  en [shape=diamond, label="enable"];
  suc [shape=diamond, label="success"];

  scan [shape=box3d, label="ScanCtrl\ncol 0..10, row 0..10, done"];
  shift [shape=box3d, label="shift_I[12]"];
  adj [label="Adj\nsticky, 8-neighbor"];
  rowc [label="RowCount[2]\nreused every 11"];
  cols [label="11 × ColCount[2]"];
  regs [label="11 × RegionCount[2]"];
  pop [label="PopCnt[8]"];
  bad [label="BadRow sticky"];
  u28 [label="U28  AND of all flags"];

  en -> scan;
  I -> shift;
  scan -> shift [label="enable & !done"];
  shift -> adj [label="taps 1,10,11,12"];
  I -> rowc; scan -> rowc;
  I -> cols; scan -> cols;
  I -> regs; scan -> regs;
  I -> pop;  scan -> pop;
  rowc -> bad;
  adj -> u28; bad -> u28; rowc -> u28;
  cols -> u28; regs -> u28; pop -> u28; scan -> u28;
  u28 -> suc;
}
'''
(GENERATED / 'blocks-named.dot').write_text(named)
!dot -Tsvg generated/blocks-named.dot -o generated/blocks-named.svg


<img src="generated/blocks-named.svg" alt="named block diagram" width="950">

That's the reconstructed function. Time to stop reverse-engineering and actually use it.


## 12. Solve the grid, then ask the original gates

Brute-forcing `2^121` boards would repeat the original mistake. Each row, column, and region wants exactly two stars; placing a star blocks its eight neighbors. Propagate forced groups, then branch on the group with the fewest remaining combinations.


In [ ]:
from itertools import combinations
from math import comb

rows = [set(range(row * 11, (row + 1) * 11)) for row in range(11)]
columns = [set(range(column, 121, 11)) for column in range(11)]
regions = [
    {
        row * 11 + column
        for row in range(11) for column in range(11)
        if region_map[row][column] == region
    }
    for region in range(11)
]
groups = rows + columns + regions

neighbors = []
for position in range(121):
    row, column = divmod(position, 11)
    neighbors.append({
        other_row * 11 + other_column
        for other_row in range(max(0, row - 1), min(11, row + 2))
        for other_column in range(max(0, column - 1), min(11, column + 2))
        if (other_row, other_column) != (row, column)
    })

search_nodes = 0

def place(stars, blocked, cells):
    stars, blocked = set(stars), set(blocked)
    for cell in cells:
        if cell in blocked:
            return None
        stars.add(cell)
        blocked.update(neighbors[cell])
    return stars, blocked

def solve(stars=frozenset(), blocked=frozenset()):
    global search_nodes
    search_nodes += 1
    stars, blocked = set(stars), set(blocked)

    while True:
        changed = False
        for group in groups:
            have = len(group & stars)
            candidates = group - stars - blocked
            need = 2 - have
            if need < 0 or len(candidates) < need:
                return None
            if need == 0:
                old_size = len(blocked)
                blocked.update(candidates)
                changed |= len(blocked) != old_size
            elif len(candidates) == need:
                result = place(stars, blocked, candidates)
                if result is None:
                    return None
                new_stars, new_blocked = result
                changed |= new_stars != stars or new_blocked != blocked
                stars, blocked = new_stars, new_blocked
        if not changed:
            break

    if len(stars) == 22:
        return stars if all(len(group & stars) == 2 for group in groups) else None

    choices = []
    for group in groups:
        need = 2 - len(group & stars)
        candidates = group - stars - blocked
        if need > 0:
            choices.append((comb(len(candidates), need), len(candidates), need, candidates))
    _, _, need, candidates = min(choices)

    for selection in combinations(sorted(candidates), need):
        if any(right in neighbors[left] for left, right in combinations(selection, 2)):
            continue
        result = place(stars, blocked, selection)
        if result is None:
            continue
        next_stars, next_blocked = result
        next_blocked.update(candidates - set(selection))
        answer = solve(next_stars, next_blocked)
        if answer is not None:
            return answer
    return None

solution = solve()
assert solution is not None
print('search nodes:', search_nodes)
solution_bits = bits_at(*sorted(solution))
for row in range(11):
    print(solution_bits[row * 11:(row + 1) * 11])
show_grid(solution_bits)


In [ ]:
write_grid_dot(GENERATED / 'solution.dot', region_map, solution)
!neato -n2 -Tsvg generated/solution.dot -o generated/solution.svg


<img src="generated/solution.svg" alt="solved 11x11 star battle" width="420">

A pretty grid is not proof. Feed the candidate back into the **original** extracted gate graph — the one that does not know our box names.


In [ ]:
byte_values, success_values = replay_attempt(solution_bits, trailing_edges=20)
message = bytes(byte_values).split(b'\0', 1)[0].decode('ascii')
print('success on first post-input edge:', success_values[0])
print('O[7:0] message:', repr(message))


The extracted circuit raises `success` and emits:

```text
(* TWO STARS *)
```

Geometry → nets → a handful of named boxes → a constraint system → back through the untouched gates. That's the loop closing.


## 13. Loose ends worth not lying about

**The output bytes are a second, smaller circuit.** After `done`, `U26` enables a 4-bit free-running counter (`dfxtp`, no reset) that walks an 8-flop message machine (the async-set `dfstp`s come out of reset as 1s). That's why the sample waveform says `TRY AGAIN` even though `success` is 0, and why a passing board says `(* TWO STARS *)`. We never needed the ROM contents to find the board; we needed `success`.

**A failed recognizer is not proof of absence.** `shift_I` was almost rejected because its stages sit on four clock-tree leaves. Walk through the buffers; they are the same `clk`.

**Don't zero every flop and call it reset.** `dfstp` sets. `dfxtp` doesn't listen.

**`n0550` really is floating.** Geometry shows a routed conductor that only touches two gate inputs. It has no path to `success`. It *can* change `O[1]` and `O[4]` for some arbitrary states — which is not the same as "for some state reachable after reset."

**Current-state influence is not reachable-state influence.** Worth keeping in your pocket the next time a floating net looks scary.

**Do not over-credit the hint.** "The night sky awaits" suggested stars. Transaction length, the scan controller, the taps, the two-bit family, and gate-level replay are what actually established the function.


In [ ]:
!.venv/bin/python -m tools.check_influence {ListOfNets} n0550 n0526
!.venv/bin/python -m tools.check_influence {ListOfNets} n0550 'O[1]'
!.venv/bin/python -m tools.check_influence {ListOfNets} n0550 success


## 14. If we have to do this again

1. Inventory cells and pins; extract nets; refuse to proceed with undiagnosed floating messes (except the one you documented).
2. Simulate at clock edges. Replay any supplied trace before believing a story.
3. Collapse the obvious repeated motif (here: the mux/DFF chain on `I`). Draw its **interface**, not its internals.
4. Cut remaining flops at Q, build the D-cone graph, collapse SCCs. That's the new schematic.
5. Characterize boxes in data-flow order. The machine that ignores the data pin is the cursor; don't name grid neighbors until you have a cursor.
6. Ask each leftover singleton "whose D-cone is only this small set of already-named boxes?" Sticky flags fall out.
7. Families of identical tiny SCCs get one representative experiment, then a map (who twitches when `I` flips).
8. Replace bit-level search with the semantic constraints you just earned.
9. Verify the candidate on the original gates, not on the cartoon.

A large sequential circuit is rarely one 92-bit mystery. It is counters, windows, sticky bits, and a final AND. The work is finding the seams.
